## Immunofluorescence Baseline Morphology Analysis
### Author: Eleni Aretaki
### Date: 2026-03-30
### Purpose
Visualize baseline morphological differences across KO cell lines using:
- Nuclei Count
- Nuclei Area

Analysis includes:
- DMSO-only conditions
- Three biological replicates
- Panels 1–3 combined (shared acquisition settings)
- Panel 4 analyzed separately (not part of the paper figure)
- Barplots with replicate overlay and FDR-adjusted significance

No normalization or KO/WT transformation applied.

#### Cell Lines: ARID1A, ARID1B, ARID2, SMARCA4, BRD9, SMARCC1, WT

In [ ]:
# -------------------------------
# Imports
# -------------------------------
import pandas as pd
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
import string

In [ ]:
# -------------------------------
# Preprocessing Functions
# -------------------------------

# Get panel/replicate from folder name
def get_panel_and_replicate(folder_name):
    num = int(folder_name.split('_')[1])
    replicate = ((num - 45) // 4) + 1
    panel = ((num - 45) % 4) + 1
    return panel, replicate

# Convert numeric row/col to well ID like A1, B12, etc
def numeric_to_well_id(row_num, col_num):
    if pd.isnull(row_num) or pd.isnull(col_num):
        return None
    try:
        row_letter = string.ascii_uppercase[int(row_num) - 1]
        col_str = str(int(col_num))
        return f"{row_letter}{col_str}"
    except Exception as e:
        print(f"Invalid row/col values: {row_num}, {col_num} → {e}")
        return None

# Load data - Preprocess
def load_combined_data(base_dir, config_df):
    combined_data = []
    panel_folders = [f for f in os.listdir(base_dir) if f.startswith('OP_')]

    for folder in panel_folders:
        folder_path = os.path.join(base_dir, folder)
        panel_id, replicate = get_panel_and_replicate(folder)

        metadata_file = os.path.join(folder_path, f'Meta_data_{folder}.xlsx')
        if not os.path.exists(metadata_file):
            continue

        df_meta = pd.read_excel(metadata_file, sheet_name='Plate_Layout')
        metadata_subset = df_meta[['well_ID', 'cell_line_modifications', 'treatment_1_name']].copy()
        metadata_subset['well_ID'] = metadata_subset['well_ID'].str.strip()

        panel_config = config_df[config_df['panel_id'] == panel_id]
        if panel_config.empty:
            continue

        for file_name in panel_config['file_name'].unique():
            data_file_path = os.path.join(folder_path, file_name)
            if not os.path.exists(data_file_path):
                continue

            skip_rows = 8 if file_name == 'PlateResults.txt' else 9
            df_data = pd.read_csv(data_file_path, sep='\t', skiprows=skip_rows)

            # Convert Row/Column to well_ID
            df_data['well_ID'] = df_data.apply(lambda x: numeric_to_well_id(x['Row'], x['Column']), axis=1)
            df_data['well_ID'] = df_data['well_ID'].str.strip()
           
            # Merge metadata
            merged_df = pd.merge(df_data, metadata_subset, on='well_ID', how='left')

            # If not PlateResults.txt, create composite key
            if file_name != 'PlateResults.txt':
                # Check required columns exist for composite key
                required_cols = ['Row', 'Column', 'Field']

                missing_cols = [col for col in required_cols if col not in merged_df.columns]
                if missing_cols:
                    print(f"⚠️ Missing columns {missing_cols} in {file_name}, skipping composite key creation.")
                else:
                    # Check for any column that links to AllNuclei
                    allnuclei_ref_cols = [col for col in merged_df.columns if 'Object No in AllNuclei' in col]

                    if allnuclei_ref_cols:
                        object_number_col = allnuclei_ref_cols[0]  # Pick the first match
                    elif 'Object No' in merged_df.columns:
                        object_number_col = 'Object No'
                    else:
                        print(f"⚠️ No valid object number column in {file_name}, skipping composite key.")
                        object_number_col = None

                    if object_number_col:
                        merged_df['object_id'] = (
                            merged_df['well_ID'] + "_" +
                            merged_df['Field'].astype(str) + "_" +
                            merged_df[object_number_col].astype(str)
                        )
            else:
                # For PlateResults.txt no object_id key
                merged_df['object_id'] = None
            
            # Merge with metadata to get treatment and cell line
            #merged_df = pd.merge(df_data, metadata_subset, on='well_ID', how='left')
            merged_df['cell_line_modifications'] = merged_df['cell_line_modifications'].astype(str).str.split('_').str[0]
            merged_df = merged_df[~merged_df['well_ID'].isin([f"H{i}" for i in range(8, 13)])]

            # Annotate replicate and panel
            merged_df['replicate'] = replicate
            merged_df['panel_id'] = panel_id

            file_config = panel_config[panel_config['file_name'] == file_name]
            for _, row in file_config.iterrows():
                column_name = row['column_name']
                readout = row['readout']
                nuclei_type = row.get('nuclei_type', 'misc')
                normalize_by = row.get('normalize_by', None)

                if column_name not in merged_df.columns:
                    continue

                if pd.notna(normalize_by) and normalize_by in merged_df.columns:
                    merged_df['readout_value'] = merged_df[column_name] / merged_df[normalize_by]
                else:
                    merged_df['readout_value'] = merged_df[column_name]

                merged_df['readout'] = readout
                merged_df['nuclei_type'] = nuclei_type

                combined_data.append(
                    merged_df[[
                        'object_id', 'well_ID', 'cell_line_modifications', 'readout_value',
                        'treatment_1_name', 'readout', 'nuclei_type',
                        'replicate', 'panel_id'
                    ]]
                )

    if not combined_data:
        print("❌ No data found.")
        return pd.DataFrame()

    return pd.concat(combined_data, ignore_index=True)

In [ ]:
# -------------------------------
# ======= MAIN EXECUTION =======
# -------------------------------

# Base directory and config
base_dir = '../IF_Analysis'
config_path = os.path.join(base_dir, 'plot_config.csv')
config_df = pd.read_csv(config_path)

# Load once, reuse everywhere
full_df = load_combined_data(base_dir, config_df)

In [ ]:
# -------------------------------
# Define Constants
# -------------------------------

# Define order and colors
cell_line_order = ["C631", "SMARCA4", "SMARCC1", "ARID1A", "ARID1B", "ARID2", "BRD9"]

cell_line_colors = {
    "C631": "#9C9B9B", 
    "ARID1A": "#C52030",
    "ARID1B": "#8A181A",
    "ARID2": "#176533",
    "BRD9": "#D2AE2A",
    "SMARCA4": "#80519B",
    "SMARCC1": "#3569A1"
}

readouts_of_interest = [
    "Nuclei Count",
    "Nuclei Area"
]

full_df = full_df[full_df["nuclei_type"] == "Single"].copy()
full_df = full_df[full_df["readout"].isin(readouts_of_interest)].copy()

In [ ]:
# -------------------------------
# Plotting Function
# -------------------------------
def plot_dmso_barplots_with_significance(df, out_dir, panel4_only=False):
    mode = "Panel 4 only" if panel4_only else "Combined across Panels 1-2-3"
    print(f"\n📊 Generating {mode} DMSO barplots with significance...")
    
    # Filter to DMSO data, optionally exclude panel 4
    df_dmso = df[df['treatment_1_name'] == 'DMSO'].copy()
    if panel4_only:
        df_dmso = df_dmso[df_dmso['panel_id'] == 4]
    else:
        df_dmso = df_dmso[df_dmso['panel_id'] != 4]

    if df_dmso.empty:
        print("⚠️ No DMSO data found for the selected panels.")
        return

    # Collapse to replicate-level means
    df_reps = (
        df_dmso.groupby(["cell_line_modifications", "replicate", "readout", "nuclei_type"])['readout_value']
        .mean()
        .reset_index()
    )

    for (readout, nuclei_type), group in df_reps.groupby(["readout", "nuclei_type"]):
        if group.empty:
            continue

        # Define output directory
        panel_folder = '4_panel' if panel4_only else '1-2-3_panels'
        out_path = os.path.join(out_dir, 'Plots', 'DMSO', 'Significance Barplots', panel_folder, nuclei_type)
        os.makedirs(out_path, exist_ok=True)

        plt.figure(figsize=(2.5, 2.25))  
        plt.rcParams['font.family'] = 'Arial'

        # Barplot (mean ± SEM) with narrower width
        ax = sns.barplot(
            x="cell_line_modifications",
            y="readout_value",
            hue='cell_line_modifications',
            data=group,
            order=cell_line_order,
            palette=cell_line_colors,
            errorbar='sd',
            capsize=0.15
        )

        # adjust error bar linewidths afterwards
        for errbar in ax.get_lines():
            errbar.set_linewidth(0.5)
    
        # Overlay replicate points
        sns.stripplot(
            x="cell_line_modifications",
            y="readout_value",
            hue='cell_line_modifications',
            data=group,
            order=cell_line_order,
            palette=cell_line_colors,
            size=2,
            jitter=True,
            dodge=False,
            ax=ax
        )

        # --- WT reference line ---
        wt_vals = group.loc[group["cell_line_modifications"] == "C631", "readout_value"]
        wt_mean = wt_vals.mean()
        ax.axhline(wt_mean, ls="--", color="#9C9B9B", lw=0.5)

        # --- Statistics ---
        raw_pvals = []
        labels = []
        for cl in cell_line_order:
            if cl == "C631" or cl not in group["cell_line_modifications"].unique():
                continue
            ko_vals = group.loc[group["cell_line_modifications"] == cl, "readout_value"]
            if len(ko_vals) >= 2:  # need replicates
                _, pval = ttest_ind(wt_vals, ko_vals, equal_var=True)
                raw_pvals.append(pval)
                labels.append(cl)

        # Apply FDR correction
        if raw_pvals:
            _, adj_pvals, _, _ = multipletests(raw_pvals, method="fdr_bh")
            pval_dict = dict(zip(labels, adj_pvals))
        else:
            pval_dict = {}

        # --- Significance stars ---
        for cl, adj_pval in pval_dict.items():
            if adj_pval < 0.05:
                if adj_pval < 0.001:
                    stars = "***"
                elif adj_pval < 0.01:
                    stars = "**"
                else:
                    stars = "*"

                xloc = cell_line_order.index(cl)
                bar_mean = group.loc[group["cell_line_modifications"] == cl, "readout_value"].mean()
                bar_sem = group.loc[group["cell_line_modifications"] == cl, "readout_value"].sem()
                yloc = bar_mean + bar_sem + (0.05 * bar_mean)  # dynamic offset
                ax.text(
                    xloc, yloc, stars,
                    ha='center', va='bottom',
                    fontsize=10, color='black', fontweight='bold'
                )

        # --- Format axes ---
        # rename 'C631' tick label to 'WT'
        # --- Relabel C631 -> WT without warnings ---
        xticks = ax.get_xticks()
        xlabels = [tick.get_text() for tick in ax.get_xticklabels()]
        new_labels = ["WT" if lab == "C631" else lab for lab in xlabels]
        ax.set_xticks(xticks)
        ax.set_xticklabels(new_labels, rotation=90, fontsize=7)


        #plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7)
        ax.set_xlabel("Knockout", fontsize=8)
        ax.tick_params(axis='y', labelsize=7)  # optional: shrink y-axis ticks
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        if readout == "Nuclei Area":
            ylabel = "Nuclei Area (µm²)"
        elif readout == "Nuclei Count":
            ylabel = "Nuclei Count"

        plt.ylabel(ylabel, fontsize=8)

        plt.tight_layout()

        # Save one plot per readout
        safe_readout = readout.replace(' ', '_').replace('/', '-')
        filename = f"{safe_readout}.pdf"
        plt.savefig(os.path.join(out_path, filename), dpi=600)
        plt.close()

    print(f"✅ {mode} DMSO barplots with significance saved in:", out_path)

In [ ]:
out_dir = r"..\IF_Baseline_Plots"

plot_dmso_barplots_with_significance(full_df, out_dir, panel4_only=False)